# 04-1 · Fairness metrics on the pick

| | |
|---|---|
| **What it does** | scores the fair(lambda) pick from 03-3 with MinSat, Jain, Gini, MaxEnvy |
| **Proves** | mean alone hides an abandoned member; these four together do not |
| **Source** | original to this paper (`potluck/backend/services/fairness_metrics.py`), harvested into `nbio.py` |
| **Eval surface** | compare all four selectors' picks side by side on the same metrics |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## Rebuild F, S, and every selector's pick (same 4 members as 03-*)


In [3]:
# Step 1: rebuild
personas = json.load(open(nbio.DATA / "personas.json"))
group_names = ["Vihaan", "Vincent", "Vasu", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"], "taste": p["taste_vector"]})

import csv
flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        n = row.get("ingredient", "").strip().lower()
        if n: flavor_map[n] = {d: float(row.get(d, 0) or 0) for d in nbio.TASTE_DIMS}

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip(); return
        for v in node.values(): yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node: yield from iter_dish_names(v)

def infer_taste(d):
    m = [flavor_map[i] for i in flavor_map if i in d.lower()]
    return [sum(x[k] for x in m)/len(m) for k in nbio.TASTE_DIMS] if m else [0.0]*6

catalog = []
for f in sorted((nbio.DATA / "Menu").glob("*.json")):
    names = [n for n in iter_dish_names(json.loads(f.read_text())) if n]
    vecs = [infer_taste(n) for n in names]
    nz = [v for v in vecs if sum(abs(x) for x in v) > 0]
    avg = [sum(v[i] for v in nz)/len(nz) for i in range(6)] if nz else [0.0]*6
    catalog.append({"option_id": f.stem, "name": f.stem, "menu": [{"name": n} for n in names], "taste": avg})

F = nbio.build_feasible_set(catalog, members)
S = {o["option_id"]: {m["member_id"]: nbio.taste_similarity(m["taste"], o["taste"]) for m in members} for o in F}
labels = {o["option_id"]: o["name"] for o in F}
member_ids = [m["member_id"] for m in members]
print(f"F size: {len(F)}")


F size: 5


## Score every selector's pick on the four fairness metrics


In [4]:
# Step 2: nbio.min_sat / mean_sat / jain / gini / max_envy -- original to this paper
candidate_vectors = {o["option_id"]: o["taste"] for o in F}
member_vectors = [m["taste"] for m in members]

rows = []
for sel in ["average", "least_misery", "most_pleasure", "fair"]:
    ranked = nbio.rank_candidates(candidate_vectors, member_vectors, method=sel, lam=0.5)
    pick_id, _ = ranked[0]
    vals = [S[pick_id][mid] for mid in member_ids]
    rows.append({"selector": sel, "pick": labels[pick_id][:22], "mean": nbio.mean_sat(vals),
                 "min": nbio.min_sat(vals), "jain": nbio.jain(vals), "gini": nbio.gini(vals),
                 "max_envy": nbio.max_envy(S, pick_id, member_ids)})
nbio.table(rows, cols=["selector", "pick", "mean", "min", "jain", "gini", "max_envy"], width=22)


selector       pick       mean   min    jain   gini   max_envy
--------------------------------------------------------------
average        Casa.cafe  0.761  0.624  0.987  0.062  0.009   
least_misery   Casa.cafe  0.761  0.624  0.987  0.062  0.009   
most_pleasure  Casa.cafe  0.761  0.624  0.987  0.062  0.009   
fair           Casa.cafe  0.761  0.624  0.987  0.062  0.009   


## Read the metrics against each other, not alone

High Jain with high MaxEnvy would mean "everyone equally unhappy" — the failure
mode the equality metrics alone would call fair. MinSat and MaxEnvy are what
catch it.


In [5]:
# Step 3: per-member breakdown on the fair(0.5) pick
fair_pick_id, _ = nbio.rank_candidates(candidate_vectors, member_vectors, method="fair", lam=0.5)[0]
print(f"pick: {labels[fair_pick_id]}\n")
for m in members:
    v = S[fair_pick_id][m["member_id"]]
    bar = "#" * int(round(v * 40))
    print(f"   {m['display_name']:<8} {v:.3f}  {bar}")

worst = min(members, key=lambda m: S[fair_pick_id][m["member_id"]])
best_alt = max(S.items(), key=lambda kv: kv[1][worst["member_id"]])
print(f"\nworst-off: {worst['display_name']}  s_i={S[fair_pick_id][worst['member_id']]:.3f}")
print(f"their best alternative in F: {labels[best_alt[0]]}  s_i={best_alt[1][worst['member_id']]:.3f}")
print(f"their envy: {best_alt[1][worst['member_id']] - S[fair_pick_id][worst['member_id']]:.3f}")


pick: Casa.cafe

   Vihaan   0.624  #########################
   Vincent  0.746  ##############################
   Vasu     0.825  #################################
   Alice    0.849  ##################################

worst-off: Vihaan  s_i=0.624
their best alternative in F: Casa.cafe  s_i=0.624
their envy: 0.000


### Reading it

The metrics table above is Table 1's fairness columns, computed on real menu
data end to end from ingredients through to a scored pick. Read MinSat and
MaxEnvy alongside Jain, not instead of it — a selector that scores well on Jain
alone could still be leaving one member badly off if MaxEnvy is also high.

Next: **04-2** gives each member their own best safe dish and checks whether it
lifts the worst-off member's outcome.
